# 03 — Baseline modeling (classification)

**Phase:** EDA is complete. This notebook moves from **business framing** to a **defendable,
leakage-safe** baseline for predicting **pre-release commercial success** as a **three-class**
problem: *flop*, *average*, *hit* (`movie_success_class`).

**Workflow:** objective → feature policy → split → `sklearn` preprocessing **pipelines** →
baseline models → **evaluation** (with class-imbalance awareness) → **interpretation** →
executive conclusions → next iterations.

**Style:** each analytical block ends with **Observation → Business interpretation → Modeling implication**
(same rhythm as notebook 02).


## 1. Business objective

**Goal:** Predict whether a movie will land in a **commercial outcome bucket** before release:
**flop** (ROI &lt; 1), **average** (1 ≤ ROI &lt; 2), or **hit** (ROI ≥ 2) — using only information
that would plausibly exist **at greenlight / pre-release planning** time in this PoC.

**Why studios care (decision support, not magic):**
- **Greenlighting & budget calibration:** compare projects under capital constraints.
- **Marketing prioritization:** focus spend where upside/risk profile warrants it.
- **Portfolio diversification:** balance slates across genres, seasons, and risk.
- **Risk management:** surface projects likely to underperform *relative to peers* given observable inputs.

**Why classification here (vs. regressing raw ROI):**
- **Executives reason in bands:** “safe”, “ok”, “home run” maps to operational conversations better than a point forecast of ROI.
- **ROI is extremely skewed:** small errors on huge hits dominate naive regression metrics; **ordered classes** align with how finance teams talk about outcomes.
- **Robustness to outliers:** buckets dampen the influence of a handful of astronomical returns.
- **Actionability:** thresholds can be tied to investment hurdles (e.g. “breakeven vs strong return”).

**What this is / is not:**
- **Predictive decision support:** rank and screen opportunities from historical patterns.
- **Not causal inference:** we do not claim “changing budget *causes* a hit”; confounders (brand, IP, unobserved marketing) remain.

---

**Observation:** A classification target turns a volatile continuous KPI into **stakeholder language**.  
**Business interpretation:** The tool supports **conversation and prioritization**, not autonomous budgeting.  
**Modeling implication:** Optimize for **macro-F1** (and per-class recall) more than headline accuracy when classes are imbalanced.


## 2. Leakage-safe feature selection

### What information is available before release?

In a real pre-release workflow, you would **not** know box office, TMDB engagement after release,
or anything derived from realized revenue. Here we mirror that discipline on the TMDB extract.

**Leakage (must never enter `X`):** anything computed from **post-release** outcomes or audience reaction,
including **`revenue`**, **`roi`**, **`log_roi`**, and the **target** itself **`movie_success_class`**.
We also exclude obvious post-release proxies if present later (e.g. `vote_average`, `vote_count`, `popularity`)
— not in the baseline feature list below, but worth enforcing when you expand features.

**Allowed baseline signals (pre-release proxy set for this PoC):**
`budget`, `runtime`, `main_genre`, `original_language`, `release_month`, `release_quarter`,
`genre_count`, `production_company_count`, `production_country_count`, `spoken_language_count`.

Missing columns are **dropped from the feature list** with a printed notice (notebook stays runnable on partial exports).

---

**Observation:** Leakage is the fastest way to fake performance.  
**Business interpretation:** Defensible models only use inputs you would truly have when deciding *before* release.  
**Modeling implication:** Centralize `FEATURE_COLUMNS` / `TARGET_COLUMN` and print **kept vs removed** columns every run.


In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

_CWD = Path.cwd().resolve()
if (_CWD / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD
elif (_CWD.parent / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD.parent
else:
    PROJECT_ROOT = _CWD
    print("Warning: data/processed not found; using cwd as PROJECT_ROOT.")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "movies_cleaned_with_target.csv"
PLOTS_DIR = PROJECT_ROOT / "plots" / "modeling"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk", font_scale=0.95)
plt.rcParams["figure.figsize"] = (11, 5.5)
plt.rcParams["axes.titlesize"] = 14

def save_fig(name: str) -> Path:
    path = PLOTS_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    return path

TARGET_COLUMN = "movie_success_class"
CANDIDATE_FEATURES = [
    "budget",
    "runtime",
    "main_genre",
    "original_language",
    "release_month",
    "release_quarter",
    "genre_count",
    "production_company_count",
    "production_country_count",
    "spoken_language_count",
]
LEAKAGE_DROP = ["revenue", "roi", "log_roi", TARGET_COLUMN]

if not DATA_PATH.exists():
    df = None
    print(f"Missing dataset: {DATA_PATH}")
    print("Run notebooks/01_dataset_audit_and_target.ipynb first.")
else:
    df = pd.read_csv(DATA_PATH)
    print("Loaded:", DATA_PATH, "| shape:", df.shape)

if df is not None:
    present = [c for c in CANDIDATE_FEATURES if c in df.columns]
    missing = [c for c in CANDIDATE_FEATURES if c not in df.columns]
    if missing:
        print("Skipping missing feature columns:", missing)
    FEATURE_COLUMNS = present
    print("FEATURE_COLUMNS:", FEATURE_COLUMNS)
    print("TARGET_COLUMN:", TARGET_COLUMN)
    print("Leakage columns (excluded from X, listed for audit):", [c for c in LEAKAGE_DROP if c in df.columns])
else:
    FEATURE_COLUMNS = []


In [ ]:
if df is None or not FEATURE_COLUMNS:
    print("Cannot proceed without data and at least one feature column.")
else:
    y = df[TARGET_COLUMN].astype(str)
    X = df[FEATURE_COLUMNS].copy()

    for col in ["main_genre", "original_language"]:
        if col in X.columns:
            X[col] = X[col].astype("string").fillna("__missing__")

    for col in ["release_month", "release_quarter"]:
        if col in X.columns:
            X[col] = X[col].apply(lambda v: "__missing__" if pd.isna(v) else str(int(v)))

    cat_cols = [c for c in ["main_genre", "original_language", "release_month", "release_quarter"] if c in X.columns]
    num_cols = [c for c in FEATURE_COLUMNS if c not in cat_cols]

    print("Numeric columns:", num_cols)
    print("Categorical columns:", cat_cols)
    print("y value counts:\n", y.value_counts())


## 3. Train / test split

We hold out **20%** of movies as a test set (`test_size=0.2`, `random_state=42`) and **stratify** on
`movie_success_class` so each split preserves the **class mix** as much as possible.

**Why stratification matters:** With imbalanced buckets (often fewer “hits”), a random split can
accidentally yield a test fold with almost no hits — metrics become noisy and **optimistic/pessimistic**
by luck. Stratification stabilizes comparison across models.

---

**Observation:** Train and test class histograms should look similar in proportion.  
**Business interpretation:** We evaluate models on **out-of-sample** movies, closer to “future releases”.  
**Modeling implication:** Always stratify on `y` for classification unless you have a time-based split (future improvement).


In [ ]:
if df is None or not FEATURE_COLUMNS:
    print("Skip split.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )
    print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
    print("\nTrain class distribution (%):")
    display((y_train.value_counts(normalize=True) * 100).round(1).to_frame("train_%"))
    print("\nTest class distribution (%):")
    display((y_test.value_counts(normalize=True) * 100).round(1).to_frame("test_%"))


## 4. Preprocessing pipeline (`ColumnTransformer` + `Pipeline`)

We separate **numeric** and **categorical** columns and build one reproducible `sklearn` **pipeline**:

- **Numeric:** `SimpleImputer(median)` → `StandardScaler()` (scale after imputation).
- **Categorical:** `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown="ignore")` so unseen
  categories at test time do not break the pipeline.

The classifier is the final step in the same `Pipeline` — this is the standard pattern for **leak-free**
CV and deployment later (`src/data.py` can mirror this structure in a future phase).

---

**Observation:** All preprocessing is **fit on train only** inside each model pipeline.  
**Business interpretation:** Prevents “peeking” at test-set statistics (professional credibility).  
**Modeling implication:** One object (`pipe.fit(X_train, y_train)`) is the unit you serialize for production PoCs.


In [ ]:
if df is None or not FEATURE_COLUMNS:
    build_preprocessor = None
    print("Skip preprocessor.")
else:
    def build_preprocessor():
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]
        )
        transformers = []
        if num_cols:
            transformers.append(("num", numeric_transformer, num_cols))
        if cat_cols:
            transformers.append(("cat", categorical_transformer, cat_cols))
        if not transformers:
            raise ValueError("No numeric or categorical columns to preprocess.")
        return ColumnTransformer(transformers=transformers)

    preprocessor = build_preprocessor()
    print("Example preprocessor (template):", preprocessor)


## 5. Baseline models

We train **simple, interpretable** baselines on the **same** leakage-safe feature matrix:

- **Logistic regression (multinomial):** linear decision boundaries; `class_weight="balanced"` to partially offset class imbalance.
- **Random forest:** non-linear interactions; `class_weight="balanced"`.
- **Gradient boosting (sklearn):** strong default tree ensemble; *no* `class_weight` in sklearn — we keep defaults for a clean baseline.
- **XGBoost (optional):** included **only if** `xgboost` is installed; otherwise skipped with an explicit message.

Models are trained in a **loop** and results are collected in a single comparison table (consulting-style, not a sprawling experiment grid).

---

**Observation:** Different inductive biases (linear vs tree) reveal whether signal is mostly additive or interaction-driven.  
**Business interpretation:** If a simple model is “good enough”, prefer it for **transparency** with stakeholders.  
**Modeling implication:** Lock `random_state=42` for repeatability in class demos.


In [ ]:
HAS_XGB = False
try:
    from xgboost import XGBClassifier

    HAS_XGB = True
except Exception:
    XGBClassifier = None
    print("xgboost not available (import/runtime error) — skipping XGBClassifier baseline.")

if df is None or not FEATURE_COLUMNS:
    results_rows = []
    fitted_models = {}
    build_preprocessor = None
    print("Skip training.")
else:
    fitted_models = {}

    def make_lr():
        return Pipeline(
            steps=[
                ("prep", build_preprocessor()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=3000,
                        class_weight="balanced",
                        random_state=42,
                        solver="lbfgs",
                    ),
                ),
            ]
        )

    def make_rf():
        return Pipeline(
            steps=[
                ("prep", build_preprocessor()),
                (
                    "clf",
                    RandomForestClassifier(
                        n_estimators=200,
                        class_weight="balanced",
                        random_state=42,
                        n_jobs=-1,
                    ),
                ),
            ]
        )

    def make_gbc():
        return Pipeline(
            steps=[
                ("prep", build_preprocessor()),
                (
                    "clf",
                    GradientBoostingClassifier(random_state=42),
                ),
            ]
        )

    model_builders = {
        "logistic_regression": make_lr,
        "random_forest": make_rf,
        "gradient_boosting": make_gbc,
    }
    if HAS_XGB:
        def make_xgb():
            return Pipeline(
                steps=[
                    ("prep", build_preprocessor()),
                    (
                        "clf",
                        XGBClassifier(
                            n_estimators=200,
                            max_depth=5,
                            learning_rate=0.1,
                            objective="multi:softprob",
                            num_class=int(y_train.nunique()),
                            random_state=42,
                            n_jobs=-1,
                            eval_metric="mlogloss",
                        ),
                    ),
                ]
            )

        model_builders["xgboost"] = make_xgb

    results_rows = []
    for name, builder in model_builders.items():
        pipe = builder()
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        macro = f1_score(y_test, y_pred, average="macro")
        results_rows.append(
            {
                "model": name,
                "accuracy": acc,
                "macro_f1": macro,
            }
        )
        fitted_models[name] = pipe
        print(name, "| accuracy:", round(acc, 4), "| macro_f1:", round(macro, 4))

    results_df = pd.DataFrame(results_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
    display(results_df)


## 6. Evaluation

We report **accuracy**, **macro-averaged F1**, `classification_report` (per-class precision/recall/F1),
and a **confusion matrix** for the **best model by macro-F1** (primary ranking metric here).

**Why macro-F1 often matters more than accuracy for executives’ *real* question:**  
Accuracy can look “fine” while the model **ignores rare but valuable classes** (e.g. hits). Macro-F1
averages **equal weight** across flop / average / hit, forcing the model to perform on **all** segments,
closer to balanced portfolio thinking.

**Figures (saved to `plots/modeling/`):**
- Model comparison bar chart (macro-F1 + accuracy).
- Confusion matrix heatmap for the best macro-F1 model.

---

**Observation:** Lift in macro-F1 usually matters more than a few points of accuracy when classes differ in frequency.  
**Business interpretation:** You care about **recall on hits** and **recall on flops** for different decisions — the report makes gaps explicit.  
**Modeling implication:** Use macro-F1 to pick a baseline champion; refine per-class thresholds later if business costs are asymmetric.


In [ ]:
if not results_rows:
    print("Skip evaluation.")
else:
    best_name = results_df.iloc[0]["model"]
    best_pipe = fitted_models[best_name]
    y_best = best_pipe.predict(X_test)

    print("Best model (by macro_f1):", best_name)
    print("\nClassification report (test set):")
    print(classification_report(y_test, y_best, digits=3))

    fig, ax = plt.subplots(figsize=(8, 5))
    res_melt = results_df.melt(id_vars="model", value_vars=["macro_f1", "accuracy"], var_name="metric", value_name="value")
    sns.barplot(data=res_melt, x="model", y="value", hue="metric", palette="deep", ax=ax)
    ax.set_title("Baseline models — test accuracy vs macro-F1")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    save_fig("01_model_comparison_metrics")

    labels = sorted(y_test.unique())
    cm = confusion_matrix(y_test, y_best, labels=labels)
    fig, ax = plt.subplots(figsize=(7, 5.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion matrix — {best_name} (test)")
    plt.tight_layout()
    save_fig("02_confusion_matrix_best_model")


## 7. Feature importance & interpretation (tree baselines)

For **tree ensembles** fitted in this notebook, we read `feature_importances_` from the classifier step
and align them with `prep.get_feature_names_out()` from the **fitted** `ColumnTransformer` (including
one-hot expanded names).

We plot the **top 15** importances from the **best tree-based** model among {Random Forest, Gradient Boosting, XGBoost}
by macro-F1 (logistic regression has coefficients, not this importance vector — we keep the story on trees here).

---

**Observation:** Budget and genre-related one-hot dimensions often rise to the top in entertainment baselines.  
**Business interpretation:** “Scale of production” and “audience positioning” are plausible pre-release levers; timing and language may be weaker or entangled with unobserved market effects.  
**Modeling implication:** Next iteration should enrich **genre/studio** signal (with regularization) rather than chasing tiny gains on weak lone features without domain review.


In [ ]:
if not results_rows:
    print("Skip feature importance.")
else:
    tree_candidates = [m for m in ["random_forest", "gradient_boosting", "xgboost"] if m in fitted_models]
    tree_scores = (
        results_df[results_df["model"].isin(tree_candidates)]
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True)
    )
    if tree_scores.empty:
        print("No tree models available for importance.")
    else:
        best_tree = tree_scores.iloc[0]["model"]
        tree_pipe = fitted_models[best_tree]
        clf = tree_pipe.named_steps["clf"]
        prep = tree_pipe.named_steps["prep"]
        names = prep.get_feature_names_out()
        imp = clf.feature_importances_
        fi = pd.Series(imp, index=names).sort_values(ascending=False).head(15)

        fig, ax = plt.subplots(figsize=(9, 6))
        sns.barplot(x=fi.values, y=fi.index.astype(str), palette="viridis", ax=ax)
        ax.set_title(f"Top 15 feature importances — {best_tree}")
        ax.set_xlabel("Importance")
        plt.tight_layout()
        save_fig("03_feature_importance_top15")

        display(fi.to_frame("importance"))


## 8. Business conclusions

**What we learned (baseline, test-set honest):**
- The **leader by macro-F1** is named in the evaluation cell above; it is your current “consulting default” for discussion.
- **Usefulness:** At baseline, treat output as **screening / ranking**, not as a sole approval mechanism — signal is partial and the dataset is historical TMDB, not a full studio data warehouse.
- **Main drivers (typical story, confirm on your run):** budget scale and genre/language positioning often dominate; release timing may nudge predictions but can confound with unmeasured “tentpole season” effects.
- **Inherent uncertainty:** Box office is driven by marketing spend, competition, talent, IP — mostly **unobserved** here — so **residual error is structural**, not just “more trees”.

### What this means for a studio executive
- Use the model as a **second opinion** on **relative risk** among comparable projects with similar observable profiles.
- Use it to **structure questions** (“Why is this flagged as risky given genre and budget?”), not to **replace** creative and distribution judgment.
- If macro-F1 is modest, the honest pitch is: **decision support under uncertainty**, with **human-in-the-loop** approval.

---

**Observation:** Performance is bounded by missing causal drivers.  
**Business interpretation:** Value is in **transparency + triage**, not oracle accuracy.  
**Modeling implication:** Future work should emphasize **calibrated probabilities** and **cost-sensitive** decisions if classes have unequal business costs.


## 9. Next modeling iterations

Practical upgrades (keep scope controlled for the course, but credible in an oral defense):

- **Hyperparameter tuning:** `RandomizedSearchCV` / `GridSearchCV` on trees and regularization strength for logistic regression.
- **Richer genre signal:** multi-label encoding from raw `genres` JSON; target/frequency encoding with regularization for high-cardinality studios.
- **Text (optional, course-dependent):** simple TF–IDF on `overview` *if* you accept NLP scope — otherwise skip to stay “tabular-first”.
- **External features:** inflation-adjusted budgets, competitor release calendars, franchise indicators (requires extra data collection).
- **Ensembles & stacking:** blend logistic + forest + boosting with a meta-learner on out-of-fold predictions.
- **Probability calibration:** `CalibratedClassifierCV` so “30% hit” reads like a probability to executives.
- **Evaluation hygiene:** time-based split (train on older releases, test on newer) to reduce **optimistic bias** from era effects.

---

**Observation:** The strongest projects iterate on **features + evaluation protocol**, not only fancier algorithms.  
**Business interpretation:** Each iteration should map to a **decision** (greenlight threshold, budget band, marketing tier).  
**Modeling implication:** Keep pipelines leakage-safe and **version** the processed dataset when features change.
